# Reviewer checks run by scripts: negative controls, discretisation sensitivity, runtime

Summarises `negative_controls.tsv` (Reviewer 1, comment 7: label permutation and within-class feature shuffling, LOCO fold-0 training sets), `discretisation_sensitivity.tsv` (Reviewer 1, comment 10: cut range and number of bins) and `runtime.tsv` (Reviewer 1, comment 9). The scripts `run_*.py` in this folder produce these files.


In [1]:
import os, pandas as pd
numbers = []
def note(claim, value, source): numbers.append({'section': 'R5', 'claim': claim, 'value': value, 'source': source}); print(f'{claim}: {value}')
if os.path.exists('negative_controls.tsv'):
    nc = pd.read_csv('negative_controls.tsv', sep='\t')
    s = nc.groupby(['data_type', 'control']).agg(runs=('permutation', 'count'), pairs_mean=('unordered_pairs_passing', 'mean'), pairs_max=('unordered_pairs_passing', 'max'),
                                                 sig2d_mean=('features_2D_significant', 'mean'), max_total_IG=('max_total_IG', 'max')).round(1)
    display(s)
    for dt in ('taxa', 'functions', 'combined'):
        real = nc[(nc.data_type == dt) & (nc.control == 'real')].unordered_pairs_passing.iloc[0]
        for k in ('label_perm', 'within_class'):
            g = nc[(nc.data_type == dt) & (nc.control == k)]
            note(f'{dt}: synergistic pairs with real labels vs {k} (mean, max over {len(g)} permutations)', f'{real:,} vs {g.unordered_pairs_passing.mean():.1f} (max {g.unordered_pairs_passing.max()})', 'R5 negative_controls.tsv')


runs  pairs_mean  pairs_max  sig2d_mean  max_total_IG
data_type control                                                            
combined  label_perm       5         0.0          0         0.0           0.0
          real             1    183176.0     183176       611.0         538.9
          within_class     5    185291.4     185369       350.2         714.3
functions label_perm       5         0.2          1         0.4          22.5
          real             1     63538.0      63538       369.0         162.6
          within_class     5     69144.8      69179       266.6         185.8
taxa      label_perm       5         0.0          0         0.0           0.0
          real             1     31751.0      31751       237.0         474.4
          within_class     5     30645.8      30671       110.6         645.6

taxa: synergistic pairs with real labels vs label_perm (mean, max over 5 permutations): 31,751 vs 0.0 (max 0)
taxa: synergistic pairs with real labels vs within_class (mean, max over 5 permutations): 31,751 vs 30645.8 (max 30671)
functions: synergistic pairs with real labels vs label_perm (mean, max over 5 permutations): 63,538 vs 0.2 (max 1)
functions: synergistic pairs with real labels vs within_class (mean, max over 5 permutations): 63,538 vs 69144.8 (max 69179)
combined: synergistic pairs with real labels vs label_perm (mean, max over 5 permutations): 183,176 vs 0.0 (max 0)
combined: synergistic pairs with real labels vs within_class (mean, max over 5 permutations): 183,176 vs 185291.4 (max 185369)


In [2]:
if os.path.exists('discretisation_sensitivity.tsv'):
    ds = pd.read_csv('discretisation_sensitivity.tsv', sep='\t'); display(ds)
    for _, r in ds.iterrows():
        note(f'discretisation {r.config}: pairs passing / Jaccard of all pairs vs default / Jaccard top-2 % / Spearman total IG', f'{r.unordered_pairs_passing:,} / {r.jaccard_all_vs_default} / {r.jaccard_top2pct_vs_default} / {r.spearman_total_IG_shared_pairs}', 'R5 discretisation_sensitivity.tsv')
if os.path.exists('runtime.tsv'):
    rt = pd.read_csv('runtime.tsv', sep='\t'); display(rt)
    for _, r in rt.iterrows():
        note(f'runtime one MDFS run, {r.data_type} ({r.n_samples} x {r.n_features}, {r.candidate_pairs:,} candidate pairs)', f'{r.wall_clock_s:.0f} s, peak {r.peak_RSS_GB} GB', 'R5 runtime.tsv')
    print(rt.machine.iloc[0])
pd.DataFrame(numbers).to_csv('r5_numbers_scripts.tsv', sep='\t', index=False)


,config,seed,divisions,range,features_2D_significant,pair_IG_threshold_bits,unordered_pairs_passing,top2pct_pairs,jaccard_all_vs_default,jaccard_top2pct_vs_default,spearman_total_IG_shared_pairs,n_shared_pairs
0,default,42,1,default,237,21.09027,31751,635,1.000,1.000,1.000,31751
1,default_seed43,43,1,default,228,21.09027,29838,596,0.778,0.137,0.897,26953
2,range_1D,42,1,range_1D,235,21.09027,31306,626,0.962,0.964,0.990,30923
3,range_half,42,1,range_half,251,21.09027,31724,634,0.902,0.171,0.952,30099
4,divisions_2,42,2,default,262,28.88072,24797,495,0.655,0.264,0.920,22372


discretisation default: pairs passing / Jaccard of all pairs vs default / Jaccard top-2 % / Spearman total IG: 31,751 / 1.0 / 1.0 / 1.0
discretisation default_seed43: pairs passing / Jaccard of all pairs vs default / Jaccard top-2 % / Spearman total IG: 29,838 / 0.778 / 0.137 / 0.897
discretisation range_1D: pairs passing / Jaccard of all pairs vs default / Jaccard top-2 % / Spearman total IG: 31,306 / 0.962 / 0.964 / 0.99
discretisation range_half: pairs passing / Jaccard of all pairs vs default / Jaccard top-2 % / Spearman total IG: 31,724 / 0.902 / 0.171 / 0.952
discretisation divisions_2: pairs passing / Jaccard of all pairs vs default / Jaccard top-2 % / Spearman total IG: 24,797 / 0.655 / 0.264 / 0.92


,data_type,n_samples,n_features,candidate_pairs,pairs_passing_filters,wall_clock_s,peak_RSS_GB,machine
0,taxa,2834,473,111628,31751,2.7,0.26,"Apple M4 Pro, 14 cores, 26 GB RAM, macOS-26.6...."
1,functions,2834,445,98790,63538,2.3,0.25,"Apple M4 Pro, 14 cores, 26 GB RAM, macOS-26.6...."
2,combined,2834,918,420903,183176,9.2,0.42,"Apple M4 Pro, 14 cores, 26 GB RAM, macOS-26.6...."


runtime one MDFS run, taxa (2834 x 473, 111,628 candidate pairs): 3 s, peak 0.26 GB
runtime one MDFS run, functions (2834 x 445, 98,790 candidate pairs): 2 s, peak 0.25 GB
runtime one MDFS run, combined (2834 x 918, 420,903 candidate pairs): 9 s, peak 0.42 GB
Apple M4 Pro, 14 cores, 26 GB RAM, macOS-26.6.2-arm64-arm-64bit; R version 4.5.1 (2025-06-13) MDFS 1.5.5
